<a href="https://colab.research.google.com/github/matthewhawksby/colabnotebooks/blob/main/MNE_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne
import mne
import os
import sys
import pickle
import numpy as np
import pandas as pd
import glob
import importlib

import helper

#Check if we are in colab.
def in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

#Load the correct base path depending on if we are in colab or not.
if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/MyDrive/Colab Notebooks/MNE/mne_data'
else:
    base_path = os.path.join(os.getcwd(), 'mne-data', 'MNE-sample-data')

#Open pickle file and load it into 'blocking'
with open('/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/blocking.pkl', 'rb') as f:
    blocking = pickle.load(f)

In [ ]:
#Looking at MEG .fif files and plotting them.

firstFile = '/content/drive/MyDrive/Colab Notebooks/MNE/MNE_data/m_002/250414/M002_block0_raw.fif'
# Load a raw MEG file
raw = mne.io.read_raw_fif(firstFile, preload=True)

# Plot the signal
#raw.plot()

# View metadata
#print(raw.info)

# Plot power spectral density
#raw.plot_psd()

### Get all channel names
#all_channels = raw.info['ch_names']

### Print the list of channels
#print("All channels in the file:")
#for i, ch in enumerate(all_channels):
#    print(f"{i+1:3d}: {ch}")

In [ ]:
fs = 2000
### Find all STI channels
sti_channels = [ch for ch in raw.info['ch_names'] if ch.startswith('STI') or ch.startswith('STI') and ch != 'STI101']
if not sti_channels:
    raise ValueError("No STI channels found in the file")

events = mne.find_events(raw, shortest_event=1)

### Print the detected events
events = [(sp/fs, val) for sp, pre, val in events]

all_stimuli = []
count = 0
for time, val in events:
    if val >= 512:
        val -= 512
    if val >= 256:
        val -= 256
    if val == 0 or val == 60:
        continue
    if count == 0:
        all_stimuli.append([])
    all_stimuli[-1].append((time, val))
    count = (count + 1) % 3

#for item in all_stimuli:
#    v1, v2, v3 = item[0], item[1], item[2]
#    print([(v1, v2, v3, v2[0] - v1[0], v3[0] - v1[0])])

In [ ]:
# Load csv, detect presence of button box
dir = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data/"
files = glob.glob(dir + "M001_main_2025-04-04_*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Loaded {len(df)} trials from {len(files)} files.")

if df["key_resp.keys"].notna().sum() > df["buttonBox_2.keys"].notna().sum():
    use_button = False
    print("Using keyboard response")
    response_column = "key_resp.keys"
else:
    use_button = True
    print("Using buttonBox")
    response_column = "buttonBox_2.keys"


In [ ]:
# Add correct src path BEFORE importing helper
sys.path.append('/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src')

import helper
importlib.reload(helper)

tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)

response = {}

for index, row in df.iterrows():
    stimuli = str(row["audio_file"]).replace("\\", "/")
    resp = row[response_column]

    if stimuli.lower() == "nan":
        continue

    if str(resp).lower() == "nan":
        resp = -1
    else:
        resp = int(resp)
        if not use_button:
            resp = resp - 1

    response[stimuli] = {}
    response[stimuli]["response"] = resp

for stimuli in response:
    for key in ["speaker", "sibilants", "normality", "perplexity", "tgt_word", "tgt_sound"]:
        if stimuli not in all_data:
          print("Missing:", stimuli)
          print("Closest matches:", [k for k in all_data if row['audio_file'] in k])
          continue  # Skip this row
        response[stimuli][key] = all_data[stimuli][key]


    if "bot" in response[stimuli]["speaker"]:
        response[stimuli]["speaker"] = "bot"
    else:
        response[stimuli]["speaker"] = "human"

    if response[stimuli]["sibilants"] == "nonsibilant":
        sound_pair = ["f", "θ"]
    else:
        sound_pair = ["s", "ʃ"]

    resp = response[stimuli]["response"]
    if resp == -1:
        response[stimuli]["resp_correct"] = None
    elif response[stimuli]["tgt_sound"] == sound_pair[resp]:
        response[stimuli]["resp_correct"] = True
    else:
        response[stimuli]["resp_correct"] = False

In [ ]:
from collections import Counter

# Collect unmatched stimuli
unmatched = []

for stim in response:
    if stim not in all_data:
        unmatched.append(stim)

# Show summary and a preview
print(f"\n🚨 Unmatched stimuli: {len(unmatched)}")
if unmatched:
    print("Here are a few examples:")
    for stim in unmatched[:10]:
        print(" -", stim)


# -------------------------------------
# Filter: Only valid, annotated trials
# -------------------------------------
cleaned_response = {}

for stim, data in response.items():
    if stim not in all_data:
        continue  # skip if not in master stimuli list
    if "normality" not in data or "resp_correct" not in data:
        continue  # skip incomplete data
    cleaned_response[stim] = data

r = cleaned_response

# -------------------------------------
# Quick sanity check
# -------------------------------------
print(f"📊 Total entries in response: {len(response)}")
print(f"✅ Valid/filtered entries: {len(r)}\n")

# Optional peek at sample
for i, (k, v) in enumerate(r.items()):
    print(f"{k}: {v}")
    if i >= 4:
        break

# -------------------------------------
# Filtering logic
# -------------------------------------
def eval(i, cond):
    if "nor" in cond and i["normality"] != "normal":
        return False
    if "abn" in cond and i["normality"] != "abnormal":
        return False
    if "sib" in cond and i["sibilants"] != "sibilant":
        return False
    if "nos" in cond and i["sibilants"] != "nonsibilant":
        return False
    if "man" in cond and i["speaker"] != "human":
        return False
    if "bot" in cond and i["speaker"] != "bot":
        return False
    if "cor" in cond and i["resp_correct"] is not True:
        return False
    if "err" in cond and i["resp_correct"] is not False:
        return False
    return True

def count(r, cond=[]):
    return sum(1 for v in r.values() if eval(v, cond))

def rate(r, cond=[]):
    num = count(r, ["cor"] + cond)
    denom = count(r, cond)
    if denom == 0:
        print(f"[⚠️] No trials matched condition: {cond}")
        return 0.0
    return num / denom * 100

# -------------------------------------
# Breakdown
# -------------------------------------
print("📊 Normality distribution:")
print(Counter([v["normality"] for v in r.values()]))
print()

print("🧠 Total valid trials:", len(r))
print(f"✅ Overall recall: {rate(r, []):.2f}%")

def show_rate(label, cond):
    print(f"{label:<30} {rate(r, cond):>6.2f}%")

print("\n▶️ Condition breakdown:")
show_rate("Normal trials", ["nor"])
show_rate("Abnormal trials", ["abn"])
show_rate("Normal + Bot", ["nor", "bot"])
show_rate("Abnormal + Bot", ["abn", "bot"])
show_rate("Normal + Human", ["nor", "man"])
show_rate("Abnormal + Human", ["abn", "man"])

show_rate("Normal + Bot + Sibilant", ["nor", "bot", "sib"])
show_rate("Abnormal + Bot + Sibilant", ["abn", "bot", "sib"])
show_rate("Normal + Human + Sibilant", ["nor", "man", "sib"])
show_rate("Abnormal + Human + Sibilant", ["abn", "man", "sib"])

show_rate("Normal + Bot + Nonsibilant", ["nor", "bot", "nos"])
show_rate("Abnormal + Bot + Nonsibilant", ["abn", "bot", "nos"])
show_rate("Normal + Human + Nonsibilant", ["nor", "man", "nos"])
show_rate("Abnormal + Human + Nonsibilant", ["abn", "man", "nos"])


In [ ]:
#TODO::: Let's do recall calculations for M002 and M003.
#TODO::: Get .csv files for those experiments from Jetic.

In [ ]:
import os
import helper

# Flatten the blocking.pkl file
print(f"Total blocks: {len(blocking)}")
print(f"First block (sample): {blocking[0][:5]}")

stimulus_order = []
for block in blocking:
    for entry in block:
        if isinstance(entry, list):
            stimulus_order.extend([e for e in entry if isinstance(e, str)])
        elif isinstance(entry, str):
            stimulus_order.append(entry)

print("✅ Flattened stimulus count:", len(stimulus_order))
print("👀 Sample from stimulus_order:")
for i, stim in enumerate(stimulus_order[:10]):
    print(f"{i+1:2d}: {stim}")

# Load all_data from TSVs
tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)
print("✅ Entries in all_data:", len(all_data))

# Optional: show a few keys from all_data
print("\n📄 Sample keys in all_data:")
for i, key in enumerate(list(all_data.keys())[:10]):
    print(f"{i+1:2d}: {key}")

# Compare each entry
print("\n📊 Matched stimuli with metadata:")
missing = []

for stim in stimulus_order:
    if stim in all_data:
        meta = all_data[stim]
        print(f"{stim} → {meta['normality']}, {meta['sibilants']}, {meta['speaker']}")
    else:
        missing.append(stim)

print(f"\n❌ Unmatched entries: {len(missing)}")
if missing:
    print("🔍 Sample unmatched:")
    for m in missing[:5]:
        print(" -", m)



In [ ]:
import mne
import helper

# --- Load raw MEG file ---
fif_path = "/content/drive/MyDrive/Colab Notebooks/MNE/MNE_data/m_002/250414/M002_block0_raw.fif"
raw = mne.io.read_raw_fif(fif_path, preload=False)
events = mne.find_events(raw, stim_channel="STI101", shortest_event=1)

print(f"\n📦 Total events: {len(events)}\n")

# --- Show how the first few are decoded ---
print("🔍 Preview decoded events:")
for i, e in enumerate(events[:10]):
    decoded = helper.trigger_decode(e[2])
    print(f"{i+1:2d}: Trigger ID={e[2]:>3} → Decoded: {decoded}")

# --- Count word_on events ---
# BEFORE fix: use "speaker" == "word_on"
# AFTER fix: use "trigger" == "word_on"

# Try both ways just to compare
word_on_by_speaker = [e for e in events if helper.trigger_decode(e[2])["speaker"] == "word_on"]
word_on_by_trigger = [e for e in events if helper.trigger_decode(e[2]).get("trigger") == "word_on"]

print(f"\n🎯 Detected word_on events (by 'speaker' == 'word_on'): {len(word_on_by_speaker)}")
print(f"🎯 Detected word_on events (by 'trigger' == 'word_on'): {len(word_on_by_trigger)}")

# Summary
if len(word_on_by_speaker) > 0 and len(word_on_by_trigger) == 0:
    print("🟡 You're using the OLD version of helper.py (abusing 'speaker')")
elif len(word_on_by_trigger) > 0 and len(word_on_by_speaker) == 0:
    print("🟢 You're using the FIXED version of helper.py (correct use of 'trigger')")
elif len(word_on_by_trigger) == 0 and len(word_on_by_speaker) == 0:
    print("❌ No word_on events detected. Either decoding logic is wrong or triggers are missing.")
else:
    print("⚠️ Both speaker/trigger returned results — something is inconsistent.")


word_on_events = [
    e for e in events if helper.trigger_decode(e[2]).get("trigger") == "word_on"
]


In [ ]:
# Visual inspection
#mne.viz.plot_events(events, sfreq=raw.info['sfreq'])



In [ ]:


base_path2 = '/content/drive/MyDrive/Colab Notebooks/MNE/MNE_data/M001'
raw_list = []
for i in range(2):
    file_path = os.path.join(base_path2, f'M001_block{i}_raw.fif')
    raw = mne.io.read_raw_fif(file_path, preload=True, verbose='ERROR')
    raw_list.append(raw)

raw_combined = mne.concatenate_raws(raw_list, on_mismatch='ignore')
fs = int(raw_combined.info['sfreq'])

# Limit to STI101 to speed up
events = mne.find_events(raw_combined, stim_channel='STI101', shortest_event=1, verbose='ERROR')

word_on_events = []
for sample, _, code in events:
    decoded = helper.trigger_decode(code)
    if decoded["trigger"] == "word_on":
        word_on_events.append({
            "sample": sample,
            "onset_time": sample / fs,
            "trigger_code": code,
            **decoded
        })

print(f"✅ Found {len(word_on_events)} word_on events")


In [ ]:

# STEP 1: Load response CSV
csv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/src/data/M001_main_2025-04-04_14h50.48.761.csv"
df = pd.read_csv(csv_path)

# Normalize audio file paths to match TSV entries
df["audio_file"] = df["audio_file"].str.replace("\\", "/", regex=False)
df["audio_file"] = df["audio_file"].str.replace("../", "../", regex=False)

# Load TSV metadata
tsv_path = "/content/drive/MyDrive/Colab Notebooks/MNE/bci-main/stimuli/demo/*.tsv"
all_data = helper.stimuli_load(tsv_path)

# Filter to trials with valid `trigger_word_on.started`
df = df[df["trigger_word_on.started"].notna()].reset_index(drop=True)

print("📊 Unique values in 'buttonBox_2.keys':")
print(df["buttonBox_2.keys"].unique())

print("\n🔢 Value counts:")
print(df["buttonBox_2.keys"].value_counts(dropna=False))


# STEP 2: Merge into alignment records
records = []


for _, row in df.iterrows():
    stim = row["audio_file"]
    onset = float(row["trigger_word_on.started"])

    response_raw = str(row.get("buttonBox_2.keys")).strip()

    # Accept only "0" or "1"
    response_raw = row.get("buttonBox_2.keys")

    if pd.isna(response_raw):
      response = -1
    else:
      response = int(response_raw)  # either 0 or 1

    rt = row.get("trials.buttonBox_2.rt", None)

    rec = {
        "stimulus": stim,
        "onset_time": onset,
        "response": response,
        "rt": rt
    }

    # Metadata from TSVs
    meta = all_data.get(stim, {})

    # Preserve detailed speaker label
    rec["speaker"] = meta.get("speaker", None)

    rec["presentation.started"] = row.get("presentation.started", None)
    rec["presentation.stopped"] = row.get("presentation.stopped", None)
    rec["trigger_word_on.started"] = row.get("trigger_word_on.started", None)


    # Add grouped speaker_type
    if rec["speaker"] in {"bot", "bot2"}:
        rec["speaker_type"] = "bot"
    elif rec["speaker"] is not None:
        rec["speaker_type"] = "human"
    else:
        rec["speaker_type"] = None

    # Copy other metadata fields
    for key in ["sibilants", "normality", "tgt_sound"]:
        rec[key] = meta.get(key, None)


    # Correctness
    # Determine correctness
    if response == -1 or "tgt_sound" not in meta or "sibilants" not in meta:
        rec["resp_correct"] = None
    else:
        options = ["f", "θ"] if meta["sibilants"] == "nonsibilant" else ["s", "ʃ"]
        if 0 <= response < len(options):
            rec["resp_correct"] = (meta["tgt_sound"] == options[response])
        else:
            rec["resp_correct"] = None  # Invalid index


    records.append(rec)

alignment_df = pd.DataFrame(records)

# Filter to only trials with valid responses
valid_responses = alignment_df[alignment_df["response"] != -1].copy()

# Sort by onset time or keep original
valid_responses = valid_responses.sort_values("onset_time").reset_index(drop=True)

# Print a handful
for i, row in valid_responses.head(20).iterrows():
    options = ["f", "θ"] if row["sibilants"] == "nonsibilant" else ["s", "ʃ"]
    selected = row["response"]
    correct = row["resp_correct"]
    tgt = row["tgt_sound"]

    print(f"""
🧪 Trial {i+1}
Stimulus:     {row['stimulus']}
Onset Time:   {row['onset_time']:.3f} sec
Speaker:      {row['speaker']} | Sibilants: {row['sibilants']} | Normality: {row['normality']}
Target Sound: {tgt}
Options:      {options}
Response:     {selected} → “{options[selected] if 0 <= selected < len(options) else '???'}”
Correct?:     {correct}
RT:           {row['rt']}
""")

print(f"✅ Trials in aligned DataFrame: {len(alignment_df)}")
display(alignment_df)


In [ ]:
from collections import Counter

def accuracy_by(*conds):
    subset = alignment_df.copy()
    for cond in conds:
        subset = subset[subset[cond[0]] == cond[1]]
    total = len(subset)
    correct = subset["resp_correct"].sum() if total > 0 else 0
    acc = (correct / total * 100) if total else 0.0
    print(f"{' + '.join(f'{k}={v}' for k,v in conds):<40} → {acc:5.2f}% ({correct}/{total})")

accuracy_by(("normality", "normal"))
accuracy_by(("normality", "abnormal"))
accuracy_by(("sibilants", "sibilant"))
accuracy_by(("sibilants", "nonsibilant"))
accuracy_by(("speaker_type", "bot"))
accuracy_by(("speaker_type", "human"))
accuracy_by(("normality", "abnormal"), ("sibilants", "sibilant"), ("speaker_type", "human"))


In [ ]:
#Reaction Type by Normality

import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=alignment_df, x="normality", y="rt")
plt.title("Reaction Time by Normality")
plt.show()

In [ ]:
raw_list = []
for i in range(8):
    fpath = os.path.join(base_path2, f'M001_block{i}_raw.fif')
    raw = mne.io.read_raw_fif(fpath, preload=True, verbose='ERROR')
    raw_list.append(raw)

raw_combined = mne.concatenate_raws(raw_list, on_mismatch='ignore')

In [ ]:
# Ensure both columns are present and clean
valid_times = alignment_df.dropna(subset=["presentation.started", "presentation.stopped"]).copy()

# Calculate duration for each trial
valid_times["duration"] = valid_times["presentation.stopped"] - valid_times["presentation.started"]

# Now get stats
min_duration = valid_times["duration"].min()
max_duration = valid_times["duration"].max()
avg_duration = valid_times["duration"].mean()

print(f"📊 Trial durations:")
print(f"  🔹 Min:    {min_duration:.3f} sec")
print(f"  🔹 Max:    {max_duration:.3f} sec")
print(f"  🔹 Mean:   {avg_duration:.3f} sec")


In [ ]:
# === Imports (if not already run) ===
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

# === Set Parameters ===
sfreq = raw_combined.info['sfreq']
meg_picks = mne.pick_types(raw_combined.info, meg=True, stim=False, eog=False, exclude='bads')

tmin = -0.2  # seconds before word onset
tmax = 0.8   # seconds after word onset
n_samples = int((tmax - tmin) * sfreq)
n_channels = len(meg_picks)

# === Extract Epochs Aligned to Actual Word Onset ===
X = []
y = []

for _, row in alignment_df.iterrows():
    if row.get("normality") not in ["normal", "abnormal"]:
        continue
    if np.isnan(row.get("presentation.started")) or np.isnan(row.get("trigger_word_on.started")):
        continue

    # Actual word onset = presentation.started + trigger_word_on.started
    true_onset_time = row["presentation.started"] + row["trigger_word_on.started"]

    center_sample = int(true_onset_time * sfreq)
    start_sample = center_sample + int(tmin * sfreq)
    stop_sample = center_sample + int(tmax * sfreq)

    # Bounds check
    if start_sample < 0 or stop_sample > raw_combined.n_times:
        continue

    trial_data = raw_combined.get_data(picks=meg_picks, start=start_sample, stop=stop_sample)
    X.append(trial_data.flatten())
    y.append(1 if row["normality"] == "abnormal" else 0)

X = np.array(X)
y = np.array(y)

print(f"✅ Extracted {len(X)} trials using actual word onset")
print(f"📐 Each trial shape: {X[0].shape}")
print(f"🧮 Label counts: {dict(zip(*np.unique(y, return_counts=True)))}")

# === Classifier Function ===
def run_classifier(name, clf):
    pipe = make_pipeline(
        StandardScaler(),
        PCA(n_components=50),
        clf
    )

    scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    print(f"\n🧠 {name} Accuracy (5-fold CV): {scores.mean():.3f} ± {scores.std():.3f}")

    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=["Normal", "Abnormal"]))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d",
                xticklabels=["Normal", "Abnormal"], yticklabels=["Normal", "Abnormal"])
    plt.title(f"{name} Confusion Matrix")
    plt.show()

# === Run Classifiers ===
run_classifier("Logistic Regression", LogisticRegression(max_iter=1000))
run_classifier("Decision Tree", DecisionTreeClassifier(max_depth=5))
run_classifier("Random Forest", RandomForestClassifier(n_estimators=100))
run_classifier("SVM (RBF)", SVC(kernel='rbf'))
